In [2]:
import pyarrow.parquet as pq
table = pq.read_table("dataset/train-00000-of-00001.parquet")
df = table.to_pandas()
df.columns

Index(['instruction', 'input', 'output', 'source', 'persona', 'perspective',
       'category'],
      dtype='object')

In [3]:
import pandas as pd

SYSTEM_PROMPT = (
    "You are CyberCTRL, an expert cybersecurity assistant. "
    "Provide technically correct, clear, and complete answers."
)

def create_messages(row):
    user_prompt = row["instruction"].strip()
    if pd.notna(row["input"]) and row["input"].strip():
        user_prompt += "\n\n" + row["input"].strip()
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
        {
            "role": "assistant",
            "content": row["output"].strip(),
        },
    ]

In [4]:
df["messages"] = df.apply(create_messages, axis=1)
df['messages'][0]

[{'role': 'system',
  'content': 'You are CyberCTRL, an expert cybersecurity assistant. Provide technically correct, clear, and complete answers.'},
 {'role': 'user',
  'content': 'Can you explain CVE-2026-39528 in simple terms?'},
 {'role': 'assistant',
  'content': 'CVE-2026-39528 is a cybersecurity vulnerability with a CVSS score of 5.3 (MEDIUM). In simple terms, it means Missing Authorization vulnerability in WP Delicious WP Delicious delicious-recipes allows Exploiting Incorrectly Configured Access Control Security Levels.This issue affects WP Delicious: from n/a through <= 1.9.5. This matters because it affects various systems and could allow attackers to compromise systems if left unpatched.'}]

In [5]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["persona"],
)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./models/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

In [7]:
def formatting(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_ds = train_ds.map(formatting)
val_ds = val_ds.map(formatting)

Map:   0%|          | 0/156411 [00:00<?, ? examples/s]

Map:   0%|          | 0/17379 [00:00<?, ? examples/s]

In [8]:
import numpy as np

lengths = train_ds.map(
    lambda x: {"len": len(tokenizer(x["text"]).input_ids)}
)
lens = lengths["len"]
print(np.percentile(lens, [50, 75, 90, 95, 99]))
print(max(lengths["len"]))
print((np.array(lens) > 512).mean() * 100)

Map:   0%|          | 0/156411 [00:00<?, ? examples/s]

[138.  220.  295.  344.  489.9]
1948
0.6700296014986158


In [9]:
train_ds = train_ds.filter(
    lambda x: len(tokenizer(x["text"]).input_ids) <= 512
)
val_ds = val_ds.filter(
    lambda x: len(tokenizer(x["text"]).input_ids) <= 512
)

Filter:   0%|          | 0/156411 [00:00<?, ? examples/s]

Filter:   0%|          | 0/17379 [00:00<?, ? examples/s]

In [10]:
train_ds[0]["text"]

"<|im_start|>system\nYou are CyberCTRL, an expert cybersecurity assistant. Provide technically correct, clear, and complete answers.<|im_end|>\n<|im_start|>user\nWhat immediate steps should a security team take for ?<|im_end|>\n<|im_start|>assistant\nFor  (CVSS N/A, INFO), a security team should:\n1. Identify all assets running affected software: various systems\n2. Apply vendor patches immediately if available\n3. Implement virtual patching / WAF rules if patch cannot be applied\n4. Monitor for active exploitation attempts\n5. Review the following references for detailed guidance: \nDescription of vulnerability: V-Webmail 1.6.4 - 'pear_dir' Remote File Inclusion<|im_end|>\n"

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth 2026.7.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [12]:
from trl import SFTTrainer, SFTConfig
import torch

training_args = SFTConfig(
    output_dir="CyberCTRL",
    max_seq_length=max_seq_length,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    save_strategy="steps",             
    save_steps=500,                     
    save_total_limit=2,                 
    eval_strategy="steps",              
    eval_steps=500,                     
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    packing=True,
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    args=training_args,
)

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/155363 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=12):   0%|          | 0/155363 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/17285 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=12):   0%|          | 0/17285 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [14]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Map (num_proc=12):   0%|          | 0/51885 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/5779 [00:00<?, ? examples/s]

In [15]:
trainer.train(resume_from_checkpoint=True)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,885 | Num Epochs = 1 | Total steps = 6,486
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
5500,0.556445,0.606119
6000,0.619514,0.602319
6486,0.651179,0.601755


Unsloth: Restored added_tokens_decoder metadata in CyberCTRL/checkpoint-5500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in CyberCTRL/checkpoint-6000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in CyberCTRL/checkpoint-6486/tokenizer_config.json.


TrainOutput(global_step=6486, training_loss=0.13948361222258035, metrics={'train_runtime': 5138.2169, 'train_samples_per_second': 10.098, 'train_steps_per_second': 1.262, 'total_flos': 4.3960556343558144e+17, 'train_loss': 0.13948361222258035, 'epoch': 1.0})

In [16]:
model.save_pretrained("CyberCTRL_Qwen_3B")
tokenizer.save_pretrained("CyberCTRL_Qwen_3B")

Unsloth: Restored added_tokens_decoder metadata in CyberCTRL_Qwen_3B/tokenizer_config.json.


('CyberCTRL_Qwen_3B/tokenizer_config.json',
 'CyberCTRL_Qwen_3B/chat_template.jinja',
 'CyberCTRL_Qwen_3B/tokenizer.json')

In [17]:
model.save_pretrained_merged(
    "CyberCTRL_Qwen_3B_Merged",
    tokenizer,
    save_method="merged_16bit",
)

Detected local model directory: /teamspace/studios/this_studio/models/Qwen2.5-3B-Instruct


Unsloth: Restored added_tokens_decoder metadata in CyberCTRL_Qwen_3B_Merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /teamspace/studios/this_studio/.cache/huggingface/hub


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [00:03<00:03,  3.07s/it]

Copied model-00002-of-00002.safetensors from local model directory


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:13<00:00,  6.51s/it]


Copied model-00001-of-00002.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:24<00:00, 12.07s/it]


Unsloth: Merge process complete. Saved to `/teamspace/studios/this_studio/CyberCTRL_Qwen_3B_Merged`
